# Ground Truth Labeling

This notebook labels each sensor reading as **normal (0)** or **failure (1)**, using the cleaned failure-events log. It is the output of the "Ground Truth" task, and its result feeds the windowing and baseline steps.

## Inputs
| File | Description |
|---|---|
| `data_imputed.csv` | Sensor readings after missing values were handled (contains a `timestamp` column) |
| `failure_events_clean.csv` | Cleaned failure log with `start_time` and `end_time` for each failure event |

## Method
1. Load both files and convert the time columns to `datetime`.
2. Sort the sensor data by `timestamp` so the rows are in chronological order.
3. Create a `ground_truth` column and set it to `0` for every row.
4. For each failure event, set `ground_truth = 1` for all rows whose `timestamp` falls within `[start_time, end_time]` (both ends included).
5. Print the number of rows labeled for each event, so an event that labels zero rows is easy to spot.

## Output
`data_imputed.csv` with one extra column:

| Column | Meaning |
|---|---|
| `ground_truth` | `0` = normal operation, `1` = inside a failure period |

## Labeling rule
A row is labeled as a failure only if it falls inside a recorded failure interval. Overlapping events are handled naturally, because a row is set to 1 if any event covers it.

In [ ]:
import pandas as pd
sensors_df = pd.read_csv('data_imputed.csv')
failures_df = pd.read_csv('failure_events_clean.csv')

sensors_df['timestamp']    = pd.to_datetime(sensors_df['timestamp'])
failures_df['start_time']  = pd.to_datetime(failures_df['start_time'])
failures_df['end_time']    = pd.to_datetime(failures_df['end_time'])
sensors_df = sensors_df.sort_values('timestamp').reset_index(drop=True)

sensors_df['ground_truth'] = 0
for i, row in failures_df.iterrows():
    mask = sensors_df['timestamp'].between(row['start_time'], row['end_time'])
    print(f"Event {i}: {row['start_time']} → {row['end_time']} | rows labeled = {mask.sum()}")
    sensors_df.loc[mask, 'ground_truth'] = 1

## Validation Checks

This cell verifies that the labeling step worked correctly before the file is saved and handed to the next stage (windowing and baseline).

| Check | What it verifies | Why it matters |
|---|---|---|
| `ground_truth` contains only 0 and 1 | The label column has no other values (no NaN, no 2, etc.) | Required by the acceptance criteria for this task |
| At least one row labeled as failure | `ground_truth.sum() > 0` | If it is 0, no failure interval matched the data. Usually a timestamp format or timezone mismatch |
| `end_time >= start_time` for every event | Failure intervals are valid | A reversed interval would silently label nothing |
| Label distribution | Share of normal (0) vs. failure (1) rows | The failure rate should be reasonable and is recorded in the labeling rules |
| Data range vs. failures range | Compares the time span of the sensor data with the span of the failure events | Failures outside the data range cannot be labeled |

If any assertion fails, the cell stops with an error and **the file is not saved**.

### Output
`data_labeled.csv`: all columns of `data_imputed.csv` plus `ground_truth` (0 = normal, 1 = failure).

In [ ]:
assert set(sensors_df.ground_truth.unique()) <= {0, 1}
assert sensors_df.ground_truth.sum() > 0, "No rows were labeled! Check timestamp format / timezone"
assert (failures_df.end_time >= failures_df.start_time).all()
print("\nLabel distribution:\n", sensors_df.ground_truth.value_counts(normalize=True))
print("Data range:", sensors_df.timestamp.min(), "→", sensors_df.timestamp.max())
print("Failures range:", failures_df.start_time.min(), "→", failures_df.end_time.max())

sensors_df.to_csv('data_labeled.csv', index=False)